PePy

Author: Julia K. Varga <jvarga92@gmail.com>  
License: BSD 3 clause  
Code Repository: https://github.com/gezmi/pepy

# Command Line Interface

PePy includes a CLI tool for scoring structure files from the terminal.
After installing (`pip install -e .`), the `pepy` command is available.

This mirrors the functionality of the original `fast_avg_plddt_window_ipae_iptm.py` script, but uses the refactored PePy package under the hood.

In this notebook we call the CLI via Python's `subprocess` module so it works cross-platform.

In [1]:
import subprocess, sys, os, tempfile
import pandas as pd
pd.set_option('display.width', 600)
pd.set_option('display.max_columns', 12)

PYTHON = sys.executable
DATA = os.path.abspath('../../pepy/tests/data')
TMPDIR = tempfile.mkdtemp(prefix='pepy_docs_')

def run_cli(*args):
    """Run pepy CLI and print stdout/stderr."""
    cmd = [PYTHON, '-m', 'pepy.cli'] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    return r

## Help

In [2]:
run_cli('--help');

usage: pepy [-h] (-i INPUT | -l LIST | -g GLOB) -o OUTPUT [-b BINDER]
            [-r RECEPTOR] [--cb-cutoff CB_CUTOFF]
            [--all-atom-cutoff ALL_ATOM_CUTOFF]
            [--min-interface MIN_INTERFACE] [-d]
            [--confidence-threshold CONFIDENCE_THRESHOLD] [-j]
            [--require-confidence] [-c CPU]

Calculate protein complex interface metrics and confidence scores.

options:
  -h, --help            show this help message and exit
  -i, --input INPUT     Input structure file (.pdb or .cif)
  -l, --list LIST       Text file with one structure path per line
  -g, --glob GLOB       Glob pattern for structure files (e.g.
                        "results/*.pdb")
  -o, --output OUTPUT   Output TSV file
  -b, --binder BINDER   Binder chain(s), comma-separated (default: auto-detect
                        shortest)
  -r, --receptor RECEPTOR
                        Receptor chain(s), comma-separated (default: all non-
                        binder)
  --cb-cutoff CB_CUTOF

## Single File

Score a single structure. By default, only the interface is calculated (no confidence metrics):

In [3]:
out = os.path.join(TMPDIR, 'single.tsv')
run_cli(
    '-i', os.path.join(DATA, '1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'),
    '-o', out,
)
pd.read_csv(out, sep='\t')

Binder chain(s): B, receptor chain(s): A
Processing 1 file(s)...
Wrote 1 rows to C:\Users\jvarg\AppData\Local\Temp\pepy_docs_5sroc2hw\single.tsv


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,max_plddt
0,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,10,13,91.68,98.19


## With Confidence Metrics

Add `-j` (or `--confidence`) to also load and report iPTM, pTM, iPAE, and combined confidence.
PePy auto-discovers the confidence file (JSON for AF2/AF3, NPZ for ChAI) based on the structure filename:

In [4]:
out = os.path.join(TMPDIR, 'confidence.tsv')
run_cli(
    '-i', os.path.join(DATA, '1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'),
    '-o', out,
    '-j',
)
pd.read_csv(out, sep='\t')

Binder chain(s): B, receptor chain(s): A
Processing 1 file(s)...
Wrote 1 rows to C:\Users\jvarg\AppData\Local\Temp\pepy_docs_5sroc2hw\confidence.tsv


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,max_plddt,iptm,ptm,ipae,min_ipae,confidence
0,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,10,13,91.68,98.19,0.85,0.78,1.97,0.95,0.84


## Batch Processing

Process multiple files at once. Three input modes:

| Flag | Description |
|------|-------------|
| `-i` | Single file |
| `-l` | Text file with one path per line |
| `-g` | Glob pattern (e.g. `"predictions/*.pdb"`) |

### Using a glob pattern

In [5]:
out = os.path.join(TMPDIR, 'batch.tsv')
run_cli(
    '-g', os.path.join(DATA, '*.pdb'),
    '-o', out,
    '-j',
)
pd.read_csv(out, sep='\t')

Binder chain(s): B, receptor chain(s): A
Binder chain(s): B, receptor chain(s): A
Binder chain(s): B, receptor chain(s): A
Processing 3 file(s)...
Wrote 3 rows to C:\Users\jvarg\AppData\Local\Temp\pepy_docs_5sroc2hw\batch.tsv


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,max_plddt,iptm,ptm,ipae,min_ipae,confidence
0,1YCR.pdb,B,A,11,15,30.92,50.89,NaN,NaN,NaN,NaN,NaN
1,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,10,13,91.68,98.19,0.850000,0.78,1.97,0.95,0.84
2,pred.model_idx_0.pdb,B,A,9,13,92.37,97.42,0.825989,NaN,30.00,30.00,NaN


### Using a file list

Create a text file with one structure path per line, then pass it with `-l`:

In [6]:
# Create a file list
list_file = os.path.join(TMPDIR, 'files.txt')
with open(list_file, 'w') as f:
    f.write(os.path.join(DATA, '1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb') + '\n')
    f.write(os.path.join(DATA, 'pred.model_idx_0.pdb') + '\n')

out = os.path.join(TMPDIR, 'from_list.tsv')
run_cli('-l', list_file, '-o', out, '-j')
pd.read_csv(out, sep='\t')

Binder chain(s): B, receptor chain(s): A
Binder chain(s): B, receptor chain(s): A
Processing 2 file(s)...
Wrote 2 rows to C:\Users\jvarg\AppData\Local\Temp\pepy_docs_5sroc2hw\from_list.tsv


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,max_plddt,iptm,ptm,ipae,min_ipae,confidence
0,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,10,13,91.68,98.19,0.850000,0.78,1.97,0.95,0.84
1,pred.model_idx_0.pdb,B,A,9,13,92.37,97.42,0.825989,NaN,30.00,30.00,NaN


## Specifying Chains

By default, the shortest chain is assigned as binder. Use `-b` and `-r` for explicit assignment.
Both accept comma-separated chain IDs for multi-chain binder/receptor:

In [7]:
out = os.path.join(TMPDIR, 'chains.tsv')
run_cli(
    '-i', os.path.join(DATA, '1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'),
    '-o', out,
    '-b', 'B', '-r', 'A',
    '-j',
)
pd.read_csv(out, sep='\t')

Binder chain(s): B, receptor chain(s): A
Processing 1 file(s)...
Wrote 1 rows to C:\Users\jvarg\AppData\Local\Temp\pepy_docs_5sroc2hw\chains.tsv


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,max_plddt,iptm,ptm,ipae,min_ipae,confidence
0,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,10,13,91.68,98.19,0.85,0.78,1.97,0.95,0.84


## Interface Parameters

Control the two-stage interface calculation:

| Flag | Default | Description |
|------|---------|-------------|
| `--cb-cutoff` | 8.0 | CB prefilter distance (Å). Use `-1` to skip. |
| `--all-atom-cutoff` | 4.0 | All-atom refinement distance (Å). Use `-1` to skip. |
| `--min-interface` | 1 | Minimum binder residues to count as interface |
| `-d` | off | Drop binder residues with pLDDT below threshold |
| `--confidence-threshold` | 50.0 | pLDDT threshold when `-d` is used |

In [8]:
out = os.path.join(TMPDIR, 'strict.tsv')
run_cli(
    '-i', os.path.join(DATA, '1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb'),
    '-o', out,
    '--cb-cutoff', '6.0', '--all-atom-cutoff', '3.0',
    '-d', '--confidence-threshold', '70.0',
    '-j',
)
pd.read_csv(out, sep='\t')

Binder chain(s): B, receptor chain(s): A
Processing 1 file(s)...
Wrote 1 rows to C:\Users\jvarg\AppData\Local\Temp\pepy_docs_5sroc2hw\strict.tsv


,file,binder,receptor,n_binder_res,n_receptor_res,avg_plddt,max_plddt,iptm,ptm,ipae,min_ipae,confidence
0,1ycr_af2_55d19_unrelaxed_rank_001_alphafold2_m...,B,A,3,2,97.69,98.19,0.85,0.78,1.18,0.95,0.84


## Parallel Processing

Use `-c` to process multiple files in parallel (requires `joblib`; install with `pip install pepy[parallel]`):

```bash
# Use all available cores
pepy -g "predictions/*.pdb" -o results.tsv -j -c -1

# Use 4 cores
pepy -g "predictions/*.pdb" -o results.tsv -j -c 4
```

## Output Format

The output is a tab-separated file. Columns depend on whether `-j` is used:

**Always present:**

| Column | Description |
|--------|-------------|
| `file` | Input filename |
| `binder` | Binder chain(s) |
| `receptor` | Receptor chain(s) |
| `n_binder_res` | Number of binder interface residues |
| `n_receptor_res` | Number of receptor interface residues |
| `avg_plddt` | Mean pLDDT of binder interface CA atoms |
| `max_plddt` | Max pLDDT of binder interface CA atoms |

**With `-j` (confidence):**

| Column | Description |
|--------|-------------|
| `iptm` | Interface predicted TM-score |
| `ptm` | Predicted TM-score |
| `ipae` | Median PAE between binder–receptor interface |
| `min_ipae` | Min PAE between binder–receptor interface |
| `confidence` | Combined score (0.8 × iPTM + 0.2 × pTM) |

If a file fails to process, an `error` column will appear for that row.

## Comparison with Original Script

If you were using `fast_avg_plddt_window_ipae_iptm.py`, here's the mapping:

| Old flag | New flag | Notes |
|----------|----------|-------|
| `-i` | `-i` | Same |
| `-l` | `-l` | Same |
| — | `-g` | New: glob patterns |
| `-o` | `-o` | Same |
| `-p` | `-b` | Renamed: peptide → binder |
| `-r` | `-r` | Same |
| `-b` (cb cutoff) | `--cb-cutoff` | Now a long flag |
| `-j` | `-j` | Same |
| `-d` | `-d` | Same |
| `-m` | `--min-interface` | Now a long flag |
| `-c` | `-c` | Same |
| `-w`, `-e` | — | Removed (windowing removed entirely) |
| `-f` (focus) | — | Removed |
| `--convert_cif` | — | Automatic now (AF3 CIF fixing is built-in) |

In [9]:
# Clean up temp files
import shutil
shutil.rmtree(TMPDIR, ignore_errors=True)